Test 07/13/26

In [11]:
from copy import deepcopy
from random import random,choice

class PhyloNode():
    """A node on a phylogenetic tree"""
    
    #PhyloNode objects can be initiated
    #from Newick strings, or directly
    #with __init__
    
        
    def __init__(self,children=None,parent=None,\
      name = None,binary_traits=None):
      """Initiate a node on a phylogenetic tree
      children -- a list of PhyloNode objects
        descending immediately from this node
      name -- a string for the name of the node
      parent -- a single PhyloNode object for the parent
      binary_traits -- a dict of True/False traits
      """
      self.Name = name
      self.Children = []
      if children:
        self.Children.extend(children)
      self.Parent = parent
      self.BinaryTraits = deepcopy(binary_traits) or {}
      self.Extinct = False
      self.BranchLength = 0
      self.Support = None
    
    def __repr__(self):
        """Return a string representation of self"""
        return f"Node(name={repr(self.Name)}, length={self.BranchLength})"
        
        
    def isTip(self):
      """Return True if the node is a tip"""
      if not self.Children: #capture None or []
        return True
      else:
        return False

    def isRoot(self):
      """Return True if the node is the root of the whole tree"""
      if not self.Parent:
        return True
      else:
        return False

    def getDescendants(self):
        """Return a list of PhyloNodes descending from the current node"""

        if self.isTip():
            print(self.Name," is a tip ... returning []")
            return []

        descendants = self.Children or []
        for c in self.Children:
            #The set of descendants is described
            #by the descendants of all the nodes
            #immediate children.
            if not c.isTip():
                child_descendants = c.getDescendants()
                descendants.extend([c for c in child_descendants if c not in descendants])

            #Side note: this will fail on enormous trees
            #due to the recursion limit. Not normally a problem though.
        return descendants

    def getAncestors(self):
        """Return the ancestors of the given node"""
        if self.isRoot():
            return None

        ancestors = [self.Parent]
        parents_ancestors = self.Parent.getAncestors()
        if parents_ancestors:
            ancestors.extend(parents_ancestors)
        return ancestors

    def getRoot(self):
        """Return the root node"""
        curr_node = self
        while not curr_node.isRoot():
            curr_node = curr_node.Parent
        return curr_node

    def addChild(self,child):
        """Attach a child node"""
        if child not in self.Children:
            self.Children.append(child)
        child.Parent = self
 
    def addParent(self,parent):
        """Attach a parent node"""
        self.Parent = parent
        parent.Children.append(self)
    
    def getMRCA(self,other):
        """Return PhyloNode for the most recent common ancestor with other
        other -- another PhyloNode object in the same tree
        """
        most_recent_common_ancestor = None
        #Cache the ancestors of other since
        #we'll refer to it often
        other_ancestors = other.getAncestors()
        self_ancestors = self.getAncestors()
        #First check for the trivial case
        #where one node is root (but be sure they're on the same tree)
        if self.isRoot() and self in other_ancestors:
            return self

        if other.isRoot() and other in self_ancestors:
            return other

        for a in self_ancestors:
            #Note these will be in order of relatedness
            if a in other_ancestors:
                #End the loop the first time this happens
                #since we want the most
                #recent common ancestor
                most_recent_common_ancestor = a
                break

        if not most_recent_common_ancestor:
            raise ValueError("No common ancestor found for ",self.Name," and ",other.Name,\
            ". Are they on the same tree?")

        return most_recent_common_ancestor
    
    def speciate(self):
        """Add two children descending from this node"""
        if self.Children:
            raise ValueError("Internal nodes can't speciate")
        if self.Extinct:
            raise ValueError("Extinct nodes can't speciate")

        child1_name = self.Name + "A"
        child1=PhyloNode(name=child1_name,binary_traits=self.BinaryTraits)
        self.addChild(child1)
        child2_name = self.Name + "B"
        child2=PhyloNode(name=child2_name,binary_traits=self.BinaryTraits)
        self.addChild(child2)


    def update(self,speciation_chance=0.25,trait_change_chance=0.25,\
        extinction_chance=0.25):
        """Update the node by speciation, extinction or trait gain or loss"""
        if not self.isTip():
            print("Not updating node {name} - it's not a tip".format(name=self.Name))
            return None

        if random() < extinction_chance:
            print(self.Name," goes extinct!")
            self.Extinct = True

        if self.Extinct:
            print("Not updating node {name} - it's extinct".format(name=self.Name))
            return None
        print("Updating node:{name}".format(name=self.Name))


        for trait,value in self.BinaryTraits.items():
            if random() < trait_change_chance:
                self.BinaryTraits[trait] = not self.BinaryTraits[trait]
                print(self.Name," has a new trait value for ",\
                  trait,": ",self.BinaryTraits[trait],"!")

        if random() < speciation_chance:
            #Speciate
            print("Node {name} speciates!".format(name=self.Name))
            self.speciate()

In [12]:
# Monkey Patch PhyloNode

def from_newick(newick_str):
    """Construct a PhyloNode from a newick string"""
    tokens = tokenize(newick_str)
    tree = parse_newick_tokens(tokens)
    return tree
    
def from_newick_file(filepath):
    """Construct a tree from a single Newick string in a file"""
    with open(filepath) as f:
        newick_str = f.read()
    f.close()
    tree = PhyloNode.from_newick(newick_str)
    return tree

def tokenize(newick_str):
    tokens = []
    current_token = []
    is_inside_quotes = False
    
    for char in newick_str:
        # State Switch: If we see a quote, toggle our tracking variable
        if char == "'":
            is_inside_quotes = not is_inside_quotes
            current_token.append(char)
            continue
            
        # If we are inside quotes, spaces are part of the name! 
        # Keep collecting characters blindly.
        if is_inside_quotes:
            current_token.append(char)
            continue
            
        # If we are OUTSIDE quotes, standard Newick rules apply
        if char.isspace():
            continue  # Ignore random formatting spaces
            
        if char in '(),:;':
            # We hit syntax! First, save whatever name we were building
            if current_token:
                tokens.append("".join(current_token))
                current_token = []
            # Then, add the syntax character as its own token
            tokens.append(char)
        else:
            # It's just a normal letter of a name or number
            current_token.append(char)
            
    return tokens

def parse_newick_tokens(tokens):
    # Root node of the current subtree
    root = PhyloNode()
    current_node = root

    while tokens:
        token = tokens.pop(0)

        # 1. Opening parenthesis: A new subtree is beginning
        if token == '(':
            new_node = PhyloNode()
            current_node.addChild(new_node)
            # The new node becomes the current working node
            current_node = new_node

        # 2. Comma: The current node's subtree is done, start a new sibling
        elif token == ',':
            current_node = current_node.Parent
            new_node = PhyloNode()
            current_node.addChild(new_node)
            current_node = new_node

        # 3. Closing parenthesis: The current subtree is completed, move up to the parent
        elif token == ')':
            current_node = current_node.Parent

        # 4. Colon: The next token will dictate the branch length of the current node
        elif token == ':':
            # The next token is the length
            length_token = tokens.pop(0)
            current_node.BranchLength = float(length_token)

        # 5. Semicolon: The end of the entire Newick tree string
        elif token == ';':
            break

        # 6. Default: This must be the Name (or Label) of the current node
        else:
            # Handle quoted strings, stripping the single quotes
            if token.startswith("'") and token.endswith("'"):
                token = token[1:-1]
            #current_node.Name = token
            
            #Handle bootstrap scores
            # If the current node ALREADY has children, any label found here 
            # belongs to the internal node (usually a bootstrap support value).
            if len(current_node.Children) > 0:
                # store it as an internal support attribute
                current_node.Support = token
            else:
                # It's a standard leaf tip name (like Anolis_cybotes)
                current_node.Name = token

    return root

PhyloNode.from_newick = from_newick
PhyloNode.from_newick_file = from_newick_file
PhyloNode.tokenize = tokenize
PhyloNode.parse_newick_tokens = parse_newick_tokens

In [13]:
def traverse(self,order="preorder",starting_node=None):
    """Traverse the tree"""
    
    
    nodes = []
    
    if order == "preorder":
        #We don't want to do this using recursive calls as in some
        #examples because we don't want the method to fail on very large trees
        if not starting_node:
            starting_node = self.getRoot()

        #visit
        nodes.append(starting_node)
        for c in starting_node.Children:
            traversed_nodes = c.traverse(order="preorder",starting_node=c)
            nodes.extend(traversed_nodes)
        return nodes
    else:
        raise ValueError(f"Currently {order} traversal is not supported. Try a supported traversal type (e.g. order='preorder')")
PhyloNode.traverse = traverse

From wikipedia:
    
    text cases:
   ```
(,,(,));                               no nodes are named
(A,B,(C,D));                           leaf nodes are named
(A,B,(C,D)E)F;                         all nodes are named
(:0.1,:0.2,(:0.3,:0.4):0.5);           all but root node have a distance to parent
(:0.1,:0.2,(:0.3,:0.4):0.5):0.0;       all have a distance to parent
(A:0.1,B:0.2,(C:0.3,D:0.4):0.5);       distances and leaf names (popular)
(A:0.1,B:0.2,(C:0.3,D:0.4)E:0.5)F;     distances and all names
((B:0.2,(C:0.3,D:0.4)E:0.5)F:0.1)A;    a tree rooted on a leaf node (rare)
```

In [6]:
%matplotlib inline
import matplotlib.pyplot as plt

def draw(self,horizontal_space = 100, vertical_space=100):
    
    #Draw a tree in a vertical orientation
    tips = [n for n in self.traverse() if n.isTip()]
    print(tips)
    n_tips = len(tips)
    delta_x_per_tip = horizontal_space/n_tips
    longest_path = 0
    max_descending_branch_length = 0
    
    for n in tips:
        descendants = self.getDescendants()
        descending_branch_length = 0
        
   
        for d in descendants:
            parent = d.Parent
            while parent:
                if parent and parent.BranchLength:
                    descending_branch_length += parent.BranchLength
                parent = parent.Parent
        max_descending_branch_length = max(max_descending_branch_length,descending_branch_length)
        print(max_descending_branch_length)
    
    y_scaling = vertical_space/max_descending_branch_length
    print(max_descending_branch_length)
     
     
PhyloNode.draw = draw


    

In [7]:
newick_str = "(A:0.1,B:0.2,(C:0.3,D:0.4)E:0.5)F;"

#tree = PhyloNode.from_newick_file("./resources/endozoicomonas_tree.newick")
tree = PhyloNode.from_newick(newick_str)
print(tree)
tree.draw()


Node(name=None, length=0)
[Node(name='A', length=0.1), Node(name='big cat', length=0.2), Node(name='C', length=0.3), Node(name='D', length=0.4)]
1.0
1.0
1.0
1.0
1.0


In [8]:
tree.getDescendants()

[Node(name='A', length=0.1),
 Node(name='big cat', length=0.2),
 Node(name=None, length=0.5),
 Node(name='C', length=0.3),
 Node(name='D', length=0.4)]

In [10]:
print(tree.Name)

None
